In [0]:
import oracledb
import pandas as pd
import importlib.util

spec = importlib.util.spec_from_file_location(
    "db_config",
    "/Volumes/opsanalytics_adb_workspace01/default/oracle_connections/db_config.py"
)
db_config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(db_config)
get_connection = db_config.get_connection

In [0]:
# 2. Oracle applies the filter before returning data to Databricks.
oracle_jdbc_url = dbutils.secrets.get(
    scope="oao_secrets",
    key="ORACLE_JDBC_URL",
)

oracle_password = dbutils.secrets.get(
    scope="oao_secrets",
    key="OAO_PRODUCTION",
)

def oracle_reader(dbtable):
    return (
        spark.read.format("jdbc")
        .option("url", oracle_jdbc_url)
        .option("dbtable", dbtable)
        .option("user", "OAO_PRODUCTION")
        .option("password", oracle_password)
        .option("driver", "oracle.jdbc.OracleDriver")
        .option("oracle.net.ssl_server_dn_match", "true")
        .option("fetchsize", "10000")
    )

In [0]:
epic_cytology_table = 'PATH_ORDER_RESULT_V'

In [0]:
df = oracle_reader(f"OAO_PRODUCTION.{epic_cytology_table}").load()
(
    df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"opsanalytics_adb_workspace01.lab.epic_cytology_raw")
)
print(f"{epic_cytology_table}: {df.count()} rows written")